# Build the k-mer file from the genomes on Google Drive

Counts the 6-mers of every genome downloaded by `download_genomes_to_drive.ipynb`
(`My Drive/FYP/genomes_lab_v7/`, gzipped) with the project's own `experiments/genome/kmers.py`, then merges
them with the existing `kmer6_counts.npz` (24,926 genomes) into one file for all genomes:
`My Drive/FYP/kmer6_counts_all.npz`, in the format Hamza's code already reads.

- **Runtime:** CPU is enough (Runtime → Change runtime type → CPU; High-RAM gives more cores, so it runs faster).
- **Resumable:** genomes are counted in parts of 2,000, each saved to `My Drive/FYP/kmer_parts/`. If Colab stops,
  run all cells again: finished parts are skipped.
- **During the download:** you can run step 3 any time; it counts what has arrived and adds the rest next time.
  Run step 4 (merge) once the download is complete.
- **Checked:** gzipped genomes give exactly the same counts as the existing file (tested on 5 genomes).

## 1. Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Settings, and the project's kmers.py from Drive

In [ ]:
import os, shutil
FOLDER = '/content/drive/MyDrive/FYP'
GENOMES = f'{FOLDER}/genomes_lab_v7'          # the downloaded .fna.gz files
PARTS = f'{FOLDER}/kmer_parts'                # resumable parts
EXISTING = f'{FOLDER}/kmer6_counts.npz'       # the 24,926 genomes counted before
OUT = f'{FOLDER}/kmer6_counts_all.npz'        # the merged file for all genomes
WORKERS = os.cpu_count()
shutil.copy(f'{FOLDER}/code/kmers.py', '/content/kmers.py')
print(f'{WORKERS} CPU cores')

## 3. Count the k-mers
Prints a line per part of 2,000 genomes with the time left. Run again to resume or to add genomes downloaded since.

In [ ]:
!python /content/kmers.py --genomes "$GENOMES" --parts "$PARTS" --workers $WORKERS

## 4. Merge into one file
Only after the download is complete and step 3 has counted everything.

In [ ]:
!python /content/kmers.py --parts "$PARTS" --merge "$EXISTING" --out "$OUT"

## 5. Check

In [ ]:
import numpy as np
z = np.load(OUT)
print(f"{len(z['genome_id']):,} genomes, counts {z['counts'].shape}, {os.path.getsize(OUT) / 1e6:.0f} MB")

## 6. Make sure everything reached Drive

In [ ]:
drive.flush_and_unmount()
print('Done.')